In [1]:
# !pip install -U transformers
# !pip install -U datasets


# Load Dataset

In [ ]:
from datasets import load_dataset

# ds = load_dataset("AI-MO/NuminaMath-CoT")
ds = load_dataset("5CD-AI/Vietnamese-395k-meta-math-MetaMathQA-gg-translated")
ds = ds["train"].to_pandas()


In [ ]:
ds.head()


,response_en,original_question_en,type,query_en,original_question_vi,query_vi,response_vi
0,"The distance between two points $(x_1,y_1)$ an...",Gracie and Joe are choosing numbers on the com...,MATH_AnsAug,Gracie and Joe are choosing numbers on the com...,Gracie và Joe đang chọn số trên mặt phẳng phức...,Gracie và Joe đang chọn số trên mặt phẳng phức...,"Khoảng cách giữa hai điểm $(x_1,y_1)$ và $(x_2..."
1,The area of an equilateral triangle can be fou...,An equilateral triangle has an area of $64\sqr...,MATH_Rephrased,If the area of an equilateral triangle is $64\...,Một tam giác đều có diện tích $64\sqrt{3}$ $\t...,Nếu diện tích của một tam giác đều là $64\sqrt...,Diện tích của một tam giác đều có thể được tín...
2,Ping pong balls sell for $.10 each.\nJohnny bu...,Ping pong balls sell for $.10 each. Johnny bu...,GSM_FOBAR,Ping pong balls sell for $.10 each. Johnny bu...,"Quả bóng bàn được bán với giá 0,10 USD một quả...","Quả bóng bàn được bán với giá 0,10 USD một quả...","Quả bóng bàn được bán với giá 0,10 USD một quả..."
3,"If $x-5$ is a factor of $P(x)$, then $P(5)=0$....",What must be the value of the coefficient $c$ ...,MATH_Rephrased,What value should the coefficient $c$ have in ...,Giá trị của hệ số $c$ trong $P(x)=x^3+2x^2+cx+...,Hệ số $c$ phải có giá trị bao nhiêu trong đa t...,"Nếu $x-5$ là thừa số của $P(x)$, thì $P(5)=0$...."
4,The total area of the mural is 6m x 3m = 18 sq...,John paints a giant mural that is 6m by 3m. Th...,GSM_Rephrased,What is the total cost of the mural that John ...,John vẽ một bức tranh tường khổng lồ có kích t...,Tổng chi phí cho bức tranh tường mà John đã vẽ...,Tổng diện tích của bức tranh tường là 6m x 3m ...


In [ ]:
ds.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395000 entries, 0 to 394999
Data columns (total 7 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   response_en           395000 non-null  object
 1   original_question_en  395000 non-null  object
 2   type                  395000 non-null  object
 3   query_en              395000 non-null  object
 4   original_question_vi  395000 non-null  object
 5   query_vi              395000 non-null  object
 6   response_vi           395000 non-null  object
dtypes: object(7)
memory usage: 21.1+ MB


# EDA

In [2]:
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns

# plt.figure(figsize=(12, 5))
# sns.countplot(x="type", data=ds, color="lightgreen")
# plt.title("Distribution of Types")
# plt.xlabel("Type")
# plt.ylabel("Count")
# plt.show()


In [3]:
vi_queries = ds["query_vi"]
ds["query_len"] = [len(query) for query in vi_queries]

# plt.figure(figsize=(12, 5))
# plt.hist(ds["query_len"], bins=50, color="red", alpha=0.5)
# plt.title("Distribution of Query Lengths")
# plt.xlabel("Query Length")
# plt.ylabel("Count")
# plt.show()


In [4]:
response_queries = ds["response_vi"]
ds["response_len"] = [len(response) for response in response_queries]

# plt.figure(figsize=(12, 5))
# plt.hist(ds["response_len"], bins=50, color="lightblue")
# plt.title("Distribution of Response Lengths")
# plt.xlabel("Response Length")
# plt.ylabel("Count")
# plt.show()


# Data Validation

## Handle missing values

In [5]:
res = [
    i
    for i, data in ds.iterrows()
    if len(data["query_vi"]) == 0 or data["response_vi"] == 0
]
ds.drop(res, inplace=True)


## Drop outliers

In [6]:
def drop_outliers(df, column, lower_quantile=0.01, upper_quantile=0.99):
    q_low = df[column].quantile(lower_quantile)
    q_high = df[column].quantile(upper_quantile)
    df_filtered = df[(df[column] >= q_low) & (df[column] <= q_high)]
    return df_filtered


In [7]:
ds = drop_outliers(ds, "response_len", lower_quantile=0.05, upper_quantile=0.95)
ds = drop_outliers(ds, "query_len", lower_quantile=0.05, upper_quantile=0.95)


## Drop duplicates

In [9]:
ds[["query_vi", "response_vi"]].duplicated().sum()


np.int64(0)

In [8]:
ds.drop_duplicates(subset=["query_vi", "response_vi"], inplace=True)


In [10]:
ds


,response_en,original_question_en,type,query_en,original_question_vi,query_vi,response_vi,query_len,response_len
0,"The distance between two points $(x_1,y_1)$ an...",Gracie and Joe are choosing numbers on the com...,MATH_AnsAug,Gracie and Joe are choosing numbers on the com...,Gracie và Joe đang chọn số trên mặt phẳng phức...,Gracie và Joe đang chọn số trên mặt phẳng phức...,"Khoảng cách giữa hai điểm $(x_1,y_1)$ và $(x_2...",130,423
1,The area of an equilateral triangle can be fou...,An equilateral triangle has an area of $64\sqr...,MATH_Rephrased,If the area of an equilateral triangle is $64\...,Một tam giác đều có diện tích $64\sqrt{3}$ $\t...,Nếu diện tích của một tam giác đều là $64\sqrt...,Diện tích của một tam giác đều có thể được tín...,140,597
2,Ping pong balls sell for $.10 each.\nJohnny bu...,Ping pong balls sell for $.10 each. Johnny bu...,GSM_FOBAR,Ping pong balls sell for $.10 each. Johnny bu...,"Quả bóng bàn được bán với giá 0,10 USD một quả...","Quả bóng bàn được bán với giá 0,10 USD một quả...","Quả bóng bàn được bán với giá 0,10 USD một quả...",256,569
3,"If $x-5$ is a factor of $P(x)$, then $P(5)=0$....",What must be the value of the coefficient $c$ ...,MATH_Rephrased,What value should the coefficient $c$ have in ...,Giá trị của hệ số $c$ trong $P(x)=x^3+2x^2+cx+...,Hệ số $c$ phải có giá trị bao nhiêu trong đa t...,"Nếu $x-5$ là thừa số của $P(x)$, thì $P(5)=0$....",110,225
4,The total area of the mural is 6m x 3m = 18 sq...,John paints a giant mural that is 6m by 3m. Th...,GSM_Rephrased,What is the total cost of the mural that John ...,John vẽ một bức tranh tường khổng lồ có kích t...,Tổng chi phí cho bức tranh tường mà John đã vẽ...,Tổng diện tích của bức tranh tường là 6m x 3m ...,206,516
...,...,...,...,...,...,...,...,...,...
394993,Annie has 6 barrettes.\nShe has twice as many ...,"Annie has 6 barrettes, twice as many scrunchie...",GSM_Rephrased,"If Annie has 6 barrettes, twice as many scrunc...","Annie có 6 chiếc kẹp tóc, số kẹp tóc nhiều gấp...","Nếu Annie có 6 chiếc kẹp tóc, số kẹp tóc nhiều...",Annie có 6 cái kẹp tóc. Cô ấy có số dây buộc t...,238,552
394994,We know that 2 kg of potatoes costs $6.\nTo fi...,Daphney buys 5 kg of potatoes at the supermark...,GSM_FOBAR,Daphney buys x kg of potatoes at the supermark...,Daphney mua 5 kg khoai tây ở siêu thị. Nếu 2 k...,Daphney mua x kg khoai tây ở siêu thị. Nếu 2 k...,Chúng ta biết rằng 2 kg khoai tây có giá 6 USD...,198,497
394996,"To solve this problem, we need to determine th...","Aaron, Henry's brother, is 15 years old. Henry...",GSM_SV,"Aaron, Henry's brother, is 15 years old. Henry...","Aaron, anh trai của Henry, 15 tuổi. Chị gái củ...","Aaron, anh trai của Henry, 15 tuổi. Chị gái củ...","Để giải quyết vấn đề này, chúng ta cần xác địn...",191,663
394998,If each fence is 500 meters long and there are...,Emmalyn decided to paint fences in her neighbo...,GSM_AnsAug,Emmalyn decided to paint fences in her neighbo...,Emmalyn quyết định sơn hàng rào trong khu phố ...,Emmalyn quyết định sơn hàng rào trong khu phố ...,Nếu mỗi hàng rào dài 500 mét và có 50 hàng rào...,217,312


# Data Preprocessing

In [11]:
import pandas as pd
from datasets import Dataset, concatenate_datasets
from sklearn.model_selection import train_test_split
import copy

def prepare_data(
    ds,
    tokenizer,
    max_length: int = 984,
    batch_size: int = 1000,
    tools=None,
    num_proc: int = 4,
    test_size: float = 0.2,
    seed: int = 42,
):
    """Efficiently preprocess and split dataset for fine-tuning a chat model."""

    # --- 1️⃣ Convert DataFrame -> Dataset only if needed
    if isinstance(ds, pd.DataFrame):
        ds = Dataset.from_pandas(ds)

    # --- 2️⃣ Define formatting + tokenization
    def format_and_tokenize(examples):
        messages_batch = [
            [
                {
                    "role": "system",
                    "content": "Hãy suy nghĩ từng bước và trả lời câu hỏi",
                },
                {"role": "user", "content": query},
                {"role": "assistant", "content": response},
            ]
            for query, response in zip(examples["query_vi"], examples["response_vi"])
        ]

        formatted_texts = (
            tokenizer.apply_chat_template(
                messages_batch,
                tokenize=False,
                add_generation_prompt=False,
                tools=tools,
            )
            if tools
            else tokenizer.apply_chat_template(
                messages_batch,
                tokenize=False,
                add_generation_prompt=False,
            )
        )

        # Tokenize batched texts
        # tokenized = tokenizer(
        #     formatted_texts,
        #     padding="max_length",
        #     truncation=True,
        #     max_length=max_length,
        # )
        tokenized = tokenizer(
            formatted_texts,
            padding=False,  # CHANGE: dùng padding động để giảm VRAM (thay "max_length")
            truncation=True,
            max_length=max_length,  # CHANGE: giữ theo tham số truyền vào
        )

        # tokenized["labels"] = tokenized["input_ids"] # Đừng tự gán labels — để collator tự tạo sau khi pad => Collator sẽ pad input_ids rồi tự tạo labels từ input_ids đã pad, nên không còn lệch.
        return tokenized

    # --- 3️⃣ Shuffle + Split before map (saves processing time)
    ds = ds.shuffle(seed=seed)
    ds_train, ds_eval = ds.train_test_split(test_size=test_size, seed=seed).values()
    ds_temp = copy.deepcopy(ds_eval)
    # --- 4️⃣ Parallel map (multi-core)
    ds_train = ds_train.map(
        format_and_tokenize,
        batched=True,
        batch_size=batch_size,
        num_proc=num_proc,
        remove_columns=ds.column_names,
        desc="Tokenizing training dataset",
    )

    ds_eval = ds_eval.map(
        format_and_tokenize,
        batched=True,
        batch_size=batch_size,
        num_proc=num_proc,
        remove_columns=ds.column_names,
        desc="Tokenizing evaluation dataset",
    )

    # --- 5️⃣ Return both splits
    return ds_train, ds_eval


In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "model_weights(llama))/checkpoint-3918"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)


## Qwen2.5 Reasoning

In [ ]:
train_ds, eval_ds = prepare_data(ds, tokenizer)


Tokenizing training dataset (num_proc=4):   0%|          | 0/250720 [00:00<?, ? examples/s]

Tokenizing evaluation dataset (num_proc=4):   0%|          | 0/62680 [00:00<?, ? examples/s]

# Logging hugging face

In [15]:
# -------------------------
# Notebook Hugging Face Login
# -------------------------
import os
from dotenv import load_dotenv
from huggingface_hub import login, whoami
from requests.exceptions import HTTPError

# -------------------------
# 1. Định nghĩa đường dẫn
# -------------------------
# Vì file notebook nằm trong /home/guest/Projects/CS431
# nên ROOT_DIR chính là thư mục CS431
ROOT_DIR = (
    os.path.dirname(os.path.abspath(__file__))
    if "__file__" in globals()
    else os.getcwd()
)
ENV_PATH = os.path.join(ROOT_DIR, "envs", ".env")

print("📁 ENV_PATH =", ENV_PATH)
print("🔍 File tồn tại: ", os.path.exists(ENV_PATH))

# -------------------------
# 2. Load biến môi trường
# -------------------------
load_dotenv(ENV_PATH)
HF_TOKEN_READ = os.getenv("HF_TOKEN_READ")

# -------------------------
# 3. Thực hiện đăng nhập
# -------------------------
if HF_TOKEN_READ:
    try:
        print("INFO: Tìm thấy HUGGING_FACE_TOKEN. Đang đăng nhập...")
        login(token=HF_TOKEN_READ)
        user_info = whoami()
        print(f"✅ Đăng nhập Hugging Face thành công. User: {user_info['name']}")
    except HTTPError as e:
        print(f"❌ Lỗi HTTP khi đăng nhập: {e}")
    except Exception as e:
        print(f"❌ Lỗi khác khi đăng nhập: {e}")
else:
    print(
        "⚠️ Không tìm thấy HUGGING_FACE_TOKEN trong file .env. Một số model có thể yêu cầu đăng nhập."
    )

# -------------------------
# 4. In thông tin kiểm tra
# -------------------------
print("ROOT_DIR:", ROOT_DIR)
print("ENV_PATH:", ENV_PATH)


📁 ENV_PATH = /home/guest/Projects/CS431/envs/.env
🔍 File tồn tại:  True
INFO: Tìm thấy HUGGING_FACE_TOKEN. Đang đăng nhập...
✅ Đăng nhập Hugging Face thành công. User: KhoiBui
ROOT_DIR: /home/guest/Projects/CS431
ENV_PATH: /home/guest/Projects/CS431/envs/.env


# Model Training

In [14]:
# CHANGE: giảm phân mảnh GPU allocator (giúp tránh OOM do fragmentation)
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


## Args Config

In [19]:
class Config:
    # Training settings
    OUTPUT_DIR = "./model_weights(llama))"
    LOGGING_DIR = "./logs"
    EPOCHS = 2
    BATCH_SIZE = 4
    PER_DEVICE_EVAL_BATCH_SIZE = 4
    GRADIENT_ACCUMULATION_STEPS = 16
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 0.05
    MAX_GRAD_NORM = 0.4
    WARMUP_RATIO = 0.03

    # Evaluation and saving
    SAVE_STRATEGY = "steps"
    EVAL_STRATEGY = "steps"
    EVAL_STEPS = 500
    SAVE_STEPS = 500
    SAVE_TOTAL_LIMIT = 3
    LOGGING_STEPS = 10
    MAX_STEPS = -1

    # Mixed precision
    FP16 = False
    BF16 = True

    # Other settings
    LR_SCHEDULER_TYPE = "constant_with_warmup"
    REPORT_TO = "codecarbon"
    LOAD_BEST_MODEL_AT_END = True
    METRIC_FOR_BEST_MODEL = "eval_loss"
    GREATER_IS_BETTER = False
    EVAL_ACCUMULATION_STEPS = 16
    GROUP_BY_LENGTH = True


In [20]:
cfg = Config()


In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, os


model_name = "model_weights(llama))/checkpoint-3918"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)



if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token



device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

model.generation_config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False
model.enable_input_require_grads()
model.gradient_checkpointing_enable()

# (Tùy chọn) nếu đã cài flash-attn 2
try:
    model.config.attn_implementation = "flash_attention_2"
    print("Use flash_attention_2")
except:
    print("Do not use flash_attention_2")
    pass


`torch_dtype` is deprecated! Use `dtype` instead!


Use flash_attention_2


In [ ]:
# from transformers import AutoModelForCausalLM, AutoTokenizer

# model_name = "meta-llama/Llama-3.2-1B-Instruct"
# device = "cuda" # the device to load the model onto

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     torch_dtype="auto",
#     device_map="auto"
# )
# tokenizer = AutoTokenizer.from_pretrained(model_name)


## Reasoning Qwen2.5

In [21]:
from transformers import (
    TrainingArguments,
    Trainer,
    TrainerCallback,
    DataCollatorForLanguageModeling,
)
from transformers import EarlyStoppingCallback

# # Set padding token if not set
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir=cfg.OUTPUT_DIR,
    num_train_epochs=cfg.EPOCHS,
    per_device_train_batch_size=cfg.BATCH_SIZE,
    per_device_eval_batch_size=cfg.PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=cfg.GRADIENT_ACCUMULATION_STEPS,
    save_strategy=cfg.SAVE_STRATEGY,
    eval_strategy=cfg.EVAL_STRATEGY,
    eval_steps=cfg.EVAL_STEPS,
    save_steps=cfg.SAVE_STEPS,
    save_total_limit=cfg.SAVE_TOTAL_LIMIT,
    logging_steps=cfg.LOGGING_STEPS,
    learning_rate=cfg.LEARNING_RATE,
    weight_decay=cfg.WEIGHT_DECAY,
    fp16=cfg.FP16,
    bf16=cfg.BF16,
    max_grad_norm=cfg.MAX_GRAD_NORM,
    max_steps=cfg.MAX_STEPS,
    warmup_ratio=cfg.WARMUP_RATIO,
    group_by_length=cfg.GROUP_BY_LENGTH,
    lr_scheduler_type=cfg.LR_SCHEDULER_TYPE,
    report_to=cfg.REPORT_TO,
    logging_dir=cfg.LOGGING_DIR,
    load_best_model_at_end=cfg.LOAD_BEST_MODEL_AT_END,
    metric_for_best_model=cfg.METRIC_FOR_BEST_MODEL,
    greater_is_better=cfg.GREATER_IS_BETTER,
    eval_accumulation_steps=cfg.EVAL_ACCUMULATION_STEPS,
    dataloader_pin_memory=False,
    ddp_find_unused_parameters=False,
    remove_unused_columns=False,
    gradient_checkpointing=True,
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    # tokenizer=tokenizer,  #  processing_class=tokenizer, # Cach moi
    processing_class=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)


The model is already on multiple devices. Skipping the move to device specified in `args`.
[codecarbon WARNING @ 22:05:09] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 22:05:09] [setup] RAM Tracking...
[codecarbon INFO @ 22:05:09] [setup] CPU Tracking...


[codecarbon WARNING @ 22:05:10] We saw that you have a Intel(R) Core(TM) Ultra 7 265K but we don't know it. Please contact us.
[codecarbon WARNING @ 22:05:10] We will use the default power consumption of 4 W per thread for your 20 CPU, so 80W.
[codecarbon WARNING @ 22:05:10] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Linux OS detected: Please ensure RAPL files exist at /sys/class/powercap/intel-rapl/subsystem to measure CPU

[codecarbon INFO @ 22:05:10] CPU Model on constant consumption mode: Intel(R) Core(TM) Ultra 7 265K
[codecarbon WARNING @ 22:05:10] No CPU tracking mode found. Falling back on CPU load mode.
[codecarbon INFO @ 22:05:10] [setup] GPU Tracking...
[codecarbon INFO @ 22:05:10] Tracking Nvidia GPU via pynvml
[codecarbon INFO @ 22:05:10] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: cpu_load
                GPU Tracking Method: pynvml
   

# train base and save and push to hugging face

In [ ]:
trainer.train()


Step,Training Loss,Validation Loss
500,0.182200,0.283962
1000,0.177900,0.287040
1500,0.171700,0.283165
2000,0.165200,0.279993
2500,0.165600,0.272687
3000,0.159900,0.270895


[codecarbon INFO @ 22:05:50] Energy consumed for RAM : 0.000086 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 22:05:51] Delta energy consumed for CPU with cpu_load : 0.000035 kWh, power : 8.141475276800001 W
[codecarbon INFO @ 22:05:51] Energy consumed for All CPU : 0.000035 kWh
[codecarbon INFO @ 22:05:51] Energy consumed for all GPUs : 0.000829 kWh. Total GPU Power : 186.45514981959883 W
[codecarbon INFO @ 22:05:51] 0.000950 kWh of electricity and 0.000000 L of water were used since the beginning.
[codecarbon INFO @ 22:06:05] Energy consumed for RAM : 0.000167 kWh. RAM Power : 20.0 W
[codecarbon INFO @ 22:06:06] Delta energy consumed for CPU with cpu_load : 0.000032 kWh, power : 8.033190407000001 W
[codecarbon INFO @ 22:06:06] Energy consumed for All CPU : 0.000067 kWh
[codecarbon INFO @ 22:06:06] Energy consumed for all GPUs : 0.001667 kWh. Total GPU Power : 201.23742484585648 W
[codecarbon INFO @ 22:06:06] 0.001901 kWh of electricity and 0.000000 L of water were used since the beginni

In [25]:
# USERNAME = "KhoiBui"
# MODEL_NAME = "qwen2.5-math-1.5b-finetuned-vi"


# print("📁 ENV_PATH =", ENV_PATH)
# print("🔍 File tồn tại: ", os.path.exists(ENV_PATH))
# load_dotenv(ENV_PATH)

# # -------------------------
# # Lấy token write
# # -------------------------
# HF_TOKEN_WRITE = os.getenv("HF_TOKEN_WRITE")
# if not HF_TOKEN_WRITE:
#     raise ValueError("⚠️ Không tìm thấy HF_TOKEN_WRITE trong file .env")

# # -------------------------
# # Override biến môi trường HF_TOKEN
# # -------------------------
# os.environ["HF_TOKEN"] = HF_TOKEN_WRITE  # cực kỳ quan trọng
# os.environ["HF_HOME"] = os.path.expanduser("~/.cache/huggingface")  # optional

# # -------------------------
# # Login Hugging Face với quyền write
# # -------------------------
# login(token=HF_TOKEN_WRITE)
# user_info = whoami()
# print(f"✅ Đăng nhập Hugging Face với quyền write. User: {user_info['name']}")

# # -------------------------
# # Push model & tokenizer
# # -------------------------
# REPO_ID = f"{USERNAME}/{MODEL_NAME}"
# trainer.hub_model_id = REPO_ID  # gán repo trước
# trainer.push_to_hub(commit_message="Push fine-tuned model")  # không dùng exist_ok
# print(f"✅ Đã đẩy thành công! Kiểm tra tại: https://huggingface.co/{REPO_ID}")


# Evaluate Model

## Base Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "model_weights(llama))/checkpoint-3918"
device = "cuda" # the device to load the model onto

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)


`torch_dtype` is deprecated! Use `dtype` instead!


In [ ]:
import os
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")


In [21]:
sample = ds.iloc[1]
question = sample['query_vi']
response = sample['response_vi']

messages = [
    {'role': 'system', 'content': 'Hãy suy luận và đưa ra đáp án'},
    {'role': 'user', 'content': question}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False, 
    add_generation_prompt=True
)
print(text)
inputs = tokenizer(
    text,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=2048
    ).to(model.device)

res = model.generate(
    **inputs,
    pad_token_id=tokenizer.pad_token_id,
    do_sample=False,
    num_beams=1)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 17 Nov 2025

Hãy suy luận và đưa ra đáp án<|eot_id|><|start_header_id|>user<|end_header_id|>

Nếu diện tích của một tam giác đều là $64\sqrt{3}$ cm vuông, và mỗi cạnh của tam giác giảm đi 4 cm thì diện tích giảm đi bao nhiêu cm vuông?<|eot_id|><|start_header_id|>assistant<|end_header_id|>




In [ ]:
from sklearn.model_selection import train_test_split
train_ds, eval_ds = train_test_split(ds, test_size=0.1, random_state=42, shuffle=True)


In [ ]:

import torch
import time
import re
from tqdm import tqdm
from collections import defaultdict
import numpy as np
from transformers import pipeline

pipe = pipeline("text-generation", model="meta-llama/Llama-3.2-1B-Instruct")

def extract_answer(text):
    """
    Extract numerical answer or final answer from model output.
    Handles various formats like: "answer is 42", "= 42", "42", etc.
    """
    question = f'''
    Given the solution below, extract the final numerical answer (the last computed result). 
    Return only the answer, with no explanation.
    
    {text}
    '''
    
    messages = [
        {"role": "user", "content": question},
    ]
    response =pipe(messages)
    
    return response[0]['generated_text'][1]['content']

def exact_match(pred, label):
    """Check if prediction exactly matches label"""
    pred_answer = extract_answer(str(pred))
    label_answer = extract_answer(str(label))
    
    # Normalize strings
    if isinstance(pred_answer, str) and isinstance(label_answer, str):
        pred_answer = pred_answer.lower().strip()
        label_answer = label_answer.lower().strip()
    
    return pred_answer == label_answer

def numerical_accuracy(pred, label, tolerance=1e-4):
    """Check if numerical answer is within tolerance"""
    try:
        pred_num = float(extract_answer(str(pred)))
        label_num = float(extract_answer(str(label)))
        return abs(pred_num - label_num) <= tolerance
    except:
        return False

def contains_reasoning_steps(text):
    """Check if response contains reasoning steps"""
    indicators = [
        'vì', 'do đó', 'nên', 'vậy', 'because', 'therefore', 'thus',
        'step', 'bước', 'first', 'trước tiên', 'then', 'sau đó',
        '1.', '2.', 'a)', 'b)'
    ]
    
    text_lower = text.lower()
    return any(ind in text_lower for ind in indicators)

def evaluate(model, tokenizer, ds, device='cuda', batch_size=4, max_samples=None):
    
    model.eval()
    
    preds = []
    labels = []
    timings = []
    reasoning_flags = []
    
    # Limit samples if specified
    if max_samples:
        ds = ds.head(max_samples)
    
    print(f"Evaluating on {len(ds)} samples with batch_size={batch_size}")
    
    with torch.no_grad():
        for i in tqdm(range(0, len(ds), batch_size), desc="Evaluating"):
            batch = ds.iloc[i:i+batch_size]
            
            # Prepare batch messages
            messages_batch = []
            batch_labels = []
            
            for _, sample in batch.iterrows():
                question = sample['query_vi']
                response = sample['response_vi']
                
                messages = [
                    {'role': 'system', 'content': 'Hãy suy luận và đưa ra đáp án'},
                    {'role': 'user', 'content': question}
                ]
                messages_batch.append(messages)
                batch_labels.append(response)
            
            # Apply chat template
            texts = [
                tokenizer.apply_chat_template(
                    msg, 
                    tokenize=False, 
                    add_generation_prompt=True
                )
                for msg in messages_batch
            ]
            
            # Tokenize
            model_inputs = tokenizer(
                texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=1024
            ).to(device)
            
            input_length = model_inputs['input_ids'].shape[1]
            
            # Measure generation time
            start_time = time.time()
            
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=1000,
                pad_token_id=tokenizer.pad_token_id,
                do_sample=False,
                num_beams=1,
            )
            
            generation_time = time.time() - start_time
            timings.append(generation_time / len(texts))  # Time per sample
            
            # Decode
            new_tokens = generated_ids[:, input_length:]
            batch_preds = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
            
            # Store results
            preds.extend(batch_preds)
            labels.extend(batch_labels)
            
            # Check for reasoning steps
            for pred in batch_preds:
                reasoning_flags.append(contains_reasoning_steps(pred))
    
    # Calculate metrics
    metrics = calculate_metrics(preds, labels, timings, reasoning_flags)
    
    return {
        'metrics': metrics,
        'predictions': preds,
        'labels': labels,
        'timings': timings
    }

def calculate_metrics(preds, labels, timings, reasoning_flags):
    """Calculate all evaluation metrics"""
    n = len(preds)
    
    # Exact match accuracy
    print("Calculating exact match accuracy...")
    exact_matches = []
    for p, l in tqdm(zip(preds, labels), total=n, desc="Exact match"):
        try:
            exact_matches.append(exact_match(p, l))
        except Exception as e:
            print(f"Error in exact_match for pair: {e}")
            exact_matches.append(False)
    
    exact_match_acc = sum(exact_matches) / n if n > 0 else 0
    
    # Numerical accuracy (with tolerance) - only for numeric answers
    print("Calculating numerical accuracy...")
    numerical_matches = []
    numeric_count = 0
    
    for p, l in tqdm(zip(preds, labels), total=n, desc="Numerical accuracy"):
        try:
            pred_extracted = extract_answer(str(p))
            label_extracted = extract_answer(str(l))
            
            # Check if both can be converted to numbers
            if pred_extracted.isdigit() and label_extracted.isdigit():
                numeric_count += 1
                numerical_matches.append(numerical_accuracy(p, l))
        except Exception as e:
            print(f"Error in numerical_accuracy: {e}")
            continue
    
    # Calculate accuracy only over numeric answers
    numerical_acc = sum(numerical_matches) / numeric_count if numeric_count > 0 else 0
    
    # Reasoning metrics
    reasoning_rate = sum(reasoning_flags) / n if n > 0 else 0
    
    # Timing metrics
    avg_time = np.mean(timings) if timings else 0
    total_time = sum(timings)
    throughput = n / total_time if total_time > 0 else 0
    
    # Response length statistics
    response_lengths = [len(p.split()) for p in preds]
    avg_response_length = np.mean(response_lengths) if response_lengths else 0
    
    metrics = {
        'exact_match_accuracy': exact_match_acc,
        'numerical_accuracy': numerical_acc,
        'numeric_samples_count': numeric_count,
        'reasoning_rate': reasoning_rate,
        'avg_time_per_sample': avg_time,
        'total_time': total_time,
        'throughput_samples_per_sec': throughput,
        'avg_response_length_words': avg_response_length,
        'total_samples': n
    }
    
    return metrics


Device set to use cuda:0


In [ ]:
res =evaluate(model, tokenizer, eval_ds[:1000], device, 128)


Evaluating on 1000 samples with batch_size=128


Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
Evaluating: 100%|██████████| 8/8 [07:24<00:00, 55.53s/it]


Calculating exact match accuracy...


Exact match:   0%|          | 0/1000 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Exact match:   0%|          | 1/1000 [00:00<02:43,  6.10it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Exact match:   0%|          | 3/1000 [00:00<01:35, 10.49it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Exact match:   0%|          | 5/1000 [00:00<01:31, 10.93it/s]You seem to be using the pipelines

Calculating numerical accuracy...


Numerical accuracy:   0%|          | 0/1000 [00:00<?, ?it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Numerical accuracy:   0%|          | 1/1000 [00:00<02:16,  7.33it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Numerical accuracy:   0%|          | 2/1000 [00:00<02:16,  7.33it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
S

In [ ]:
res


{'metrics': {'exact_match_accuracy': 0.637,
  'numerical_accuracy': 0.715674362089915,
  'numeric_samples_count': 823,
  'reasoning_rate': 0.863,
  'avg_time_per_sample': np.float64(0.4443577600714679),
  'total_time': 3.554862080571743,
  'throughput_samples_per_sec': 281.3048656557629,
  'avg_response_length_words': np.float64(556.283),
  'total_samples': 1000},
 'predictions': ['assistant\n\nCó 3 feet trong một yard, vậy có $3^3=27$ feet khối trong một yard khối. Do đó, thể tích của hộp tính bằng thước khối là $\\frac{108}{27}=\\boxed{4}$ thước khối. Câu trả lời là: 4 thước khối. Câu trả lời là: 4 thước khối. Đáp án là: 4 thước khối. Đáp án là: 4 thước khối. Đáp án là: 4 thước khối. Đáp án là: 4 thước khối. Đáp án là: 4 thước khối. Đáp án là: 4 thước khối. Đáp án là: 4 thước khối. Đáp án là: 4 thước khối. Đáp án là: 4 thước khối. Đáp án là: 4 thước khối. Đáp án là: 4 thước khối. Đáp án là: 4 thước khối. Đáp án là: 4 thước khối. Đáp án là: 4 thước khối. Đáp án là: 4 thước khối. Đáp á